# FIFA 2026 World Cup — Prediction System
## Notebook 1: Data Collection

### Project Overview
A machine learning system that predicts every 2026 World Cup match as a 
full scoreline distribution, each team's probability of advancing through 
every round, and the eventual champion. Built on a Poisson goals model, 
XGBoost, and a Monte Carlo tournament simulator.

### What We Built in This Notebook
Complete data collection pipeline for all datasets needed to train the 
prediction model and generate 2026 World Cup forecasts.

---

### Data Sources

**1. Historical International Results**
- Source: martj42/international_results (GitHub)
- Coverage: 49,400 matches from 1872 to June 2026
- Contains: scorelines, venue, neutral flag, tournament type
- Role: Training labels for the Poisson goals model

**2. WC26 Fixture List**
- Source: openfootball/worldcup.json (GitHub)
- Coverage: All 104 matches — 72 group stage + 32 knockout
- Contains: dates, venues, groups, round, stage
- Role: Prediction targets for the simulator

**3. FIFA World Rankings (June 2026)**
- Source: Manually compiled from official FIFA rankings
- Coverage: All 48 WC teams
- Contains: rank, points, confederation
- Role: Team strength feature for predictions

**4. Elo Ratings**
- Source: Self-computed from the historical results spine
- Coverage: All 48 WC teams, updated through June 2026
- Contains: Elo rating per team
- Role: Strongest single strength feature — updates after every match,
  no gaps, perfectly aligned to results data

**5. Squad Market Values**
- Source: Transfermarkt (June 2026)
- Coverage: All 48 WC teams
- Contains: Total squad value in millions EUR
- Role: Current squad quality proxy — captures recent transfers
  and player form that rankings may lag

**6. Venue Heat Index**
- Source: Verified climate data per host city (WFAA, AccuWeather)
- Coverage: All 16 WC host venues
- Contains: Temperature, humidity, heat index (scaled 1-10)
- Role: Novel feature capturing climate stress on teams —
  European teams playing in Dallas/Houston face genuine
  physiological disadvantage vs teams from warm climates

---

### Key Design Decisions

- **Scorelines not W/D/L** — The 2026 format resolves group standings 
  using goal difference and goals scored. A W/D/L classifier cannot 
  produce these numbers so we model goals directly.

- **Elo over FIFA rankings** — Elo updates after every match vs 6x per 
  year for FIFA. No gaps, self-contained, stronger predictive signal.

- **Heat index as a feature** — Captures environmental stress not 
  reflected in any strength metric. One clean continuous feature, 
  scientifically grounded, directionally consistent.

- **No data leakage** — Every feature is computed using only information 
  available before the match it describes.

---

### Output Files (data/processed/)
| File | Description |
|---|---|
| fixtures_2026.csv | 104 WC fixtures with weather features |
| fifa_rankings_2026.csv | FIFA rankings for 48 WC teams |
| elo_ratings_2026.csv | Self-computed Elo for 48 WC teams |
| squad_values_2026.csv | Transfermarkt squad values |
| venue_heat_index.csv | Heat index for 16 host venues |

---
*Next: Notebook 2 — Feature Engineering*

In [1]:
# Install packages
import subprocess 
subprocess.run(["pip","install","pandas","numpy","pyarrow","requests"], check=True)
print('Done')

Done


In [2]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

print("Pandas version", pd.__version__)
print("numpy version", np.__version__)

Pandas version 3.0.2
numpy version 2.4.4


### We create the project folder structure

In [3]:
# Create the project folder
base = Path.home() / "wc2026"

folders = [
    base / "data" / "raw",
    base / "data" / "processed",
    base / "notebooks",
    base / "models",
    base / "outputs"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"Created:{folder}")
print("\nProject structure ready")

Created:/home/ubuntu/wc2026/data/raw
Created:/home/ubuntu/wc2026/data/processed
Created:/home/ubuntu/wc2026/notebooks
Created:/home/ubuntu/wc2026/models
Created:/home/ubuntu/wc2026/outputs

Project structure ready


### We download the first and most important dataset. This is the spine of the entire project: every international football match ever played, with scorelines.

In [4]:
# Download historical match results
import urllib.request

# Save it in folder - raw
raw_dir = Path.home() / "wc2026" / "data" / "raw"
result_path = raw_dir / "results.csv"

# Dowload from GitHub (open datset, updated daily)
url =  "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

print("Downloading.....")
urllib.request.urlretrieve(url, result_path)

print(f"Saved to: {result_path}")

Downloading.....
Saved to: /home/ubuntu/wc2026/data/raw/results.csv


In [5]:
# Load and Inspect the data
df = pd.read_csv(result_path, parse_dates = ["date"])

print("Shape", df.shape)
print("n\First 5 rows:")
df.head()

Shape (49477, 9)
n\First 5 rows:


<>:5: SyntaxWarning: invalid escape sequence '\F'
<>:5: SyntaxWarning: invalid escape sequence '\F'
/tmp/ipykernel_1090/1841537117.py:5: SyntaxWarning: invalid escape sequence '\F'
  print("n\First 5 rows:")


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [6]:
print("n\Last 5 rows:")
df.tail()

n\Last 5 rows:


<>:1: SyntaxWarning: invalid escape sequence '\L'
<>:1: SyntaxWarning: invalid escape sequence '\L'
/tmp/ipykernel_1090/2170033473.py:1: SyntaxWarning: invalid escape sequence '\L'
  print("n\Last 5 rows:")


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49476,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


In [7]:
# Underatand the data
print('Data range')
print('Earliest match', df.date.min().date())
print('Latest match', df.date.max().date())

print('n\Total matches', len(df))
print('Played matches', df.home_score.notna().sum())
print('Unplayed matches', df.home_score.isna().sum())
print('n\Tournament type (Top 10)')
print(df.tournament.value_counts().head(10))

Data range
Earliest match 1872-11-30
Latest match 2026-06-27
n\Total matches 49477
Played matches 49405
Unplayed matches 72
n\Tournament type (Top 10)
tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64


<>:6: SyntaxWarning: invalid escape sequence '\T'
<>:9: SyntaxWarning: invalid escape sequence '\T'
<>:6: SyntaxWarning: invalid escape sequence '\T'
<>:9: SyntaxWarning: invalid escape sequence '\T'
/tmp/ipykernel_1090/3234858546.py:6: SyntaxWarning: invalid escape sequence '\T'
  print('n\Total matches', len(df))
/tmp/ipykernel_1090/3234858546.py:9: SyntaxWarning: invalid escape sequence '\T'
  print('n\Tournament type (Top 10)')


### Download full WC26 fixture list - 104 matches

In [8]:
# Download WC26 fixture
import urllib.request
import json

# Save to raw folder
raw_dir = Path.home() / "wc2026" / "data" / "raw"
fixture_path = raw_dir / "wc2026_fixture.json"

# Download from openfootball — open source, updated daily during the tournament
url = "https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json"

print("Downloading fixture....")
urllib.request.urlretrieve(url,fixture_path)
print(f"Saved to {fixture_path}")

# Look and peek inside 
with open(fixture_path) as f:
    raw = json.load(f)

print("\nkeys in file:", list(raw.keys()))
print("Total matches:", len(raw["matches"]))
print("\nFirst match:")
raw['matches'][0]

Saved to /home/ubuntu/wc2026/data/raw/wc2026_fixture.json

keys in file: ['name', 'matches']
Total matches: 104

First match:


{'round': 'Matchday 1',
 'date': '2026-06-11',
 'time': '13:00 UTC-6',
 'team1': 'Mexico',
 'team2': 'South Africa',
 'group': 'Group A',
 'ground': 'Mexico City'}

In [9]:
# Explore the fixture structure
matches = raw['matches']

# First knockout match (no group field)
print("=== First Group match ===")
for m in matches:
    if m.get('group'):
        print(m)
        break
print("=== First Knockout match ===")
for m in matches:
    if not m.get('group'):
        print(m)
        break

print("\n=== Last match (The Final) ===")
print(matches[-1])

=== First Group match ===
{'round': 'Matchday 1', 'date': '2026-06-11', 'time': '13:00 UTC-6', 'team1': 'Mexico', 'team2': 'South Africa', 'group': 'Group A', 'ground': 'Mexico City'}
=== First Knockout match ===
{'round': 'Round of 32', 'num': 73, 'date': '2026-06-28', 'time': '12:00 UTC-7', 'team1': '2A', 'team2': '2B', 'ground': 'Los Angeles (Inglewood)'}

=== Last match (The Final) ===
{'round': 'Final', 'date': '2026-07-19', 'time': '15:00 UTC-4', 'team1': 'W101', 'team2': 'W102', 'ground': 'New York/New Jersey (East Rutherford)'}


In [10]:
# Convert fixture to Dataframe
records = []

for m in matches:
    records.append({
        "match_num":     m.get("num"),
        "date":          m.get("date"),
        "round":         m.get("round"),
        "group":         m.get("group"),
        "home_team":     m.get("team1"),
        "away_team":     m.get("team2"),
        "venue":         m.get("ground"),
        "home_score":    m.get("score1"),
        "away_score":    m.get("score2"),
    })
fixture = pd.DataFrame(records)
fixture['date'] = pd.to_datetime(fixture['date'])

print("Total fixtures", len(fixture))
print("\nFixture by round:")
print(fixture.groupby('round').size())
print("\nSample:")
fixture.head(10)

Total fixtures 104

Fixture by round:
round
Final                     1
Match for third place     1
Matchday 1                2
Matchday 10               4
Matchday 11               4
Matchday 12               4
Matchday 13               4
Matchday 14               6
Matchday 15               6
Matchday 16               6
Matchday 17               6
Matchday 2                2
Matchday 3                4
Matchday 4                4
Matchday 5                4
Matchday 6                4
Matchday 7                4
Matchday 8                4
Matchday 9                4
Quarter-final             4
Round of 16               8
Round of 32              16
Semi-final                2
dtype: int64

Sample:


,match_num,date,round,group,home_team,away_team,venue,home_score,away_score
0,NaN,2026-06-11,Matchday 1,Group A,Mexico,South Africa,Mexico City,None,None
1,NaN,2026-06-11,Matchday 1,Group A,South Korea,Czech Republic,Guadalajara (Zapopan),None,None
2,NaN,2026-06-18,Matchday 8,Group A,Czech Republic,South Africa,Atlanta,None,None
3,NaN,2026-06-18,Matchday 8,Group A,Mexico,South Korea,Guadalajara (Zapopan),None,None
4,NaN,2026-06-24,Matchday 14,Group A,Czech Republic,Mexico,Mexico City,None,None
5,NaN,2026-06-24,Matchday 14,Group A,South Africa,South Korea,Monterrey (Guadalupe),None,None
6,NaN,2026-06-12,Matchday 2,Group B,Canada,Bosnia & Herzegovina,Toronto,None,None
7,NaN,2026-06-13,Matchday 3,Group B,Qatar,Switzerland,San Francisco Bay Area (Santa Clara),None,None
8,NaN,2026-06-18,Matchday 8,Group B,Switzerland,Bosnia & Herzegovina,Los Angeles (Inglewood),None,None
9,NaN,2026-06-18,Matchday 8,Group B,Canada,Qatar,Vancouver,None,None


In [11]:
# Check match numbers
print("Matches with a number (Knockout):", fixture["match_num"].notna().sum())
print("Matches without a number (Group):", fixture["match_num"].isna().sum())

print("\nFirst few Knockout matches:")
fixture[fixture["match_num"].notna()][["match_num","round","home_team","away_team","venue"]].head(10)

Matches with a number (Knockout): 30
Matches without a number (Group): 74

First few Knockout matches:


,match_num,round,home_team,away_team,venue
72,73.0,Round of 32,2A,2B,Los Angeles (Inglewood)
73,74.0,Round of 32,1E,3A/B/C/D/F,Boston (Foxborough)
74,75.0,Round of 32,1F,2C,Monterrey (Guadalupe)
75,76.0,Round of 32,1C,2F,Houston
76,77.0,Round of 32,1I,3C/D/F/G/H,New York/New Jersey (East Rutherford)
77,78.0,Round of 32,2E,2I,Dallas (Arlington)
78,79.0,Round of 32,1A,3C/E/F/H/I,Mexico City
79,80.0,Round of 32,1L,3E/H/I/J/K,Atlanta
80,81.0,Round of 32,1D,3B/E/F/I/J,San Francisco Bay Area (Santa Clara)
81,82.0,Round of 32,1G,3A/E/H/I/J,Seattle


In [12]:
# Investigate the mismatch
print("Rounds and their counts:")
print(fixture.groupby("round").size().to_string())

print("\nKnockout matches without a match number:")
knockout_rounds = ["Round of 32", "Round of 16", "Quarter-final", "Semi-final", "Third-place", "Final"]
knockouts = fixture[fixture["round"].isin(knockout_rounds)]
print(knockouts[knockouts["match_num"].isna()][["round","home_team","away_team","venue"]])

Rounds and their counts:
round
Final                     1
Match for third place     1
Matchday 1                2
Matchday 10               4
Matchday 11               4
Matchday 12               4
Matchday 13               4
Matchday 14               6
Matchday 15               6
Matchday 16               6
Matchday 17               6
Matchday 2                2
Matchday 3                4
Matchday 4                4
Matchday 5                4
Matchday 6                4
Matchday 7                4
Matchday 8                4
Matchday 9                4
Quarter-final             4
Round of 16               8
Round of 32              16
Semi-final                2

Knockout matches without a match number:
     round home_team away_team                                  venue
103  Final      W101      W102  New York/New Jersey (East Rutherford)


In [13]:
# Add a stage column
def classify_stage(round_str):
    if not round_str:
        return "Unknown"
    r = round_str.lower()
    if "matchday" in r:    return "Group"
    if "round of 32" in r:  return "Round of 32"
    if "round of 16" in r:  return "Round of 16"
    if "quarter" in r:      return "Quarter-final"
    if "semi" in r:         return "Semi-final"
    if "third" in r:        return "Third-place"
    if "final" in r:        return "Final"
    return "Knockout"

fixture["stage"] = fixture["round"].apply(classify_stage)

print("Fixture by stage:")
print(fixture.groupby("stage").size().to_string())

Fixture by stage:
stage
Final             1
Group            72
Quarter-final     4
Round of 16       8
Round of 32      16
Semi-final        2
Third-place       1


In [14]:
# Find missing match number 
knockouts = fixture[fixture["stage"] != "Group"]

print("Total Knockout matches:", len(knockouts))
print("\nMissing match numbers:")
print(knockouts[knockouts["match_num"].isna()][["round","home_team","away_team","venue"]])

Total Knockout matches: 32

Missing match numbers:
                     round home_team away_team  \
102  Match for third place      L101      L102   
103                  Final      W101      W102   

                                     venue  
102                  Miami (Miami Gardens)  
103  New York/New Jersey (East Rutherford)  


In [15]:
# Assign match number to 3rd place amtch and final
fixture.loc[fixture["round"] == "Match for third place", "match_num"] = 103
fixture.loc[fixture["round"] == "final", "match_num"] = 104

# Conver match number to integer
fixture["match_num"] = fixture["match_num"].astype("Int64")

print("Missing match numbers now:", fixture["match_num"].isna().sum())
print("\nLast 5 matches:")
fixture.tail()[["match_num","stage","round","home_team","away_team","venue"]].to_string(index=False)

Missing match numbers now: 73

Last 5 matches:


' match_num         stage                 round home_team away_team                                 venue\n       100 Quarter-final         Quarter-final       W95       W96                           Kansas City\n       101    Semi-final            Semi-final       W97       W98                    Dallas (Arlington)\n       102    Semi-final            Semi-final       W99      W100                               Atlanta\n       103   Third-place Match for third place      L101      L102                 Miami (Miami Gardens)\n      <NA>         Final                 Final      W101      W102 New York/New Jersey (East Rutherford)'

In [16]:
# Use index as match number 
fixture["match_num"] = fixture.index + 1

print("Missing match numbers:", fixture["match_num"].isna().sum())
print("\nLast 5 matches:")
print(fixture.tail()[["match_num","stage","round","home_team","away_team","venue"]].to_string(index=False))

Missing match numbers: 0

Last 5 matches:
 match_num         stage                 round home_team away_team                                 venue
       100 Quarter-final         Quarter-final       W95       W96                           Kansas City
       101    Semi-final            Semi-final       W97       W98                    Dallas (Arlington)
       102    Semi-final            Semi-final       W99      W100                               Atlanta
       103   Third-place Match for third place      L101      L102                 Miami (Miami Gardens)
       104         Final                 Final      W101      W102 New York/New Jersey (East Rutherford)


In [17]:
fixture.tail()

,match_num,date,round,group,home_team,away_team,venue,home_score,away_score,stage
99,100,2026-07-11,Quarter-final,NaN,W95,W96,Kansas City,None,None,Quarter-final
100,101,2026-07-14,Semi-final,NaN,W97,W98,Dallas (Arlington),None,None,Semi-final
101,102,2026-07-15,Semi-final,NaN,W99,W100,Atlanta,None,None,Semi-final
102,103,2026-07-18,Match for third place,NaN,L101,L102,Miami (Miami Gardens),None,None,Third-place
103,104,2026-07-19,Final,NaN,W101,W102,New York/New Jersey (East Rutherford),None,None,Final


In [18]:
# Save fixture to disk
processed_dir = Path.home() / "wc2026" / "data" / "processed"
processed_dir.mkdir(parents = True, exist_ok = True)

fixture.to_csv(processed_dir / "fixture_2026.csv", index = False)

print("Saved")
print("Shape", fixture.shape)
print("Columns:", fixture.columns.to_list())

Saved
Shape (104, 10)
Columns: ['match_num', 'date', 'round', 'group', 'home_team', 'away_team', 'venue', 'home_score', 'away_score', 'stage']


## Weather Feature — Venue Heat Index

### Why add this?
The 2026 World Cup is hosted across 16 cities in the USA, Mexico, and Canada
during June and July. Summer conditions vary dramatically — Dallas and Houston
regularly hit 38°C with high humidity, while Seattle and San Francisco stay
cool and dry. This is not a neutral tournament environment.

### The scientific logic
Heat and humidity together cause physiological stress — reduced sprint capacity,
faster fatigue, higher heart rate, and slower recovery. This is well documented
in sports science literature. We combine temperature and humidity into a single
**Heat Index** (also called "feels like" temperature) using the standard
Rothfusz regression formula used by meteorologists worldwide.

### Why it helps the model
A European team like England (average summer temp ~18°C) playing in Dallas
(heat index ~47°C) faces a genuine performance disadvantage that neither
FIFA rankings nor Elo ratings capture — those measure historical strength,
not environmental stress on matchday.

The feature we add is:
**heat_disadvantage = venue_heat_index - team_heat_tolerance**

A positive number = team is playing somewhere hotter than they are used to.
A negative number = team is comfortable or in cooler conditions than home.

### Why it won't backfire
- We are not inventing data — heat index is a standard meteorological formula
- We are not overfitting — one simple continuous feature, not dozens
- It only affects WC26 predictions, not the historical training data
- Worst case it adds zero signal and the model ignores it
- Best case it explains upsets that pure strength metrics miss —
  for example a Gulf state team outperforming expectations in Texas heat

### Limitation
We use average June/July climate data per city, not actual matchday forecasts.
Once the tournament starts we can update with real forecast data for accuracy.

In [19]:
# Venue Heat Index (Temperature + Humidity combined)
# Heat Index formula gives "feels like" temperature
# All values are June/July averages for each host city

def heat_index(temp_c, humidity):
    """
    Calculates heat index in Celsius.
    temp_c = average temperature in Celsius
    humidity = relative humidity as percentage (e.g. 65)
    """
    # Convert to Fahrenheit for the standard formula
    T = (temp_c * 9/5) + 32
    R = humidity
    
    HI = (-42.379 + 2.04901523*T + 10.14333127*R
          - 0.22475541*T*R - 0.00683783*T*T
          - 0.05481717*R*R + 0.00122874*T*T*R
          + 0.00085282*T*R*R - 0.00000199*T*T*R*R)
    
    # Convert back to Celsius
    return round((HI - 32) * 5/9, 1)

# Venue data — avg temp and humidity in June/July
venue_data = {
    "Dallas (Arlington)":                    {"temp": 32, "humidity": 67},  # 102-108°F verified
    "Houston":                               {"temp": 33, "humidity": 75},  # hotter and more humid
    "Monterrey (Guadalupe)":                             {"temp": 33, "humidity": 55},  # hot, drier than Houston
    "Miami (Miami Gardens)":                 {"temp": 31, "humidity": 78},  # humid tropical
    "Atlanta":                               {"temp": 31, "humidity": 68},
    "Kansas City":                           {"temp": 31, "humidity": 64},
    "Philadelphia":                          {"temp": 29, "humidity": 65},
    "New York/New Jersey (East Rutherford)": {"temp": 28, "humidity": 62},
    "Boston (Foxborough)":                   {"temp": 27, "humidity": 63},
    "Guadalajara (Zapopan)":                {"temp": 27, "humidity": 50},
    "Los Angeles (Inglewood)":               {"temp": 26, "humidity": 55},
    "Toronto":                               {"temp": 25, "humidity": 63},
    "Mexico City":                           {"temp": 22, "humidity": 45},
    "Seattle":                               {"temp": 21, "humidity": 58},
    "Vancouver":                             {"temp": 20, "humidity": 60},
    "San Francisco Bay Area (Santa Clara)":  {"temp": 18, "humidity": 72},
}

# Calculate heat index for each venue
for city, data in venue_data.items():
    data["heat_index"] = heat_index(data["temp"], data["humidity"])

# Build DataFrame
venue_df = pd.DataFrame([
    {"venue": city, 
     "temp_c": data["temp"], 
     "humidity": data["humidity"],
     "heat_index": data["heat_index"]}
    for city, data in venue_data.items()
]).sort_values("heat_index", ascending=False).reset_index(drop=True)

print("Venue heat index table:")
print(venue_df.to_string(index=False))

Venue heat index table:
                                venue  temp_c  humidity  heat_index
                              Houston      33        75        45.7
                Miami (Miami Gardens)      31        78        40.2
                   Dallas (Arlington)      32        67        39.3
                Monterrey (Guadalupe)      33        55        37.8
                              Atlanta      31        68        37.0
                          Kansas City      31        64        35.9
                         Philadelphia      29        65        31.8
New York/New Jersey (East Rutherford)      28        62        29.7
                  Boston (Foxborough)      27        63        28.3
                Guadalajara (Zapopan)      27        50        27.4
              Los Angeles (Inglewood)      26        55        26.7
                              Toronto      25        63        26.0
                          Mexico City      22        45        25.0
                        

In [20]:
# Save venue heat index to processed folder
processed_dir = Path.home() / "wc2026" / "data" / "processed"

venue_df.to_csv(processed_dir / "venue_heat_index.csv", index=False)
print("Saved venue heat index!")

# Quick check - join venues to fixtures to make sure all venues match
fixture_venues = fixture["venue"].unique()
heat_venues = venue_df["venue"].values

missing = [v for v in fixture_venues if v not in heat_venues]
print(f"\nFixture venues not in heat table: {len(missing)}")
if missing:
    print("Missing:", missing)
else:
    print("All fixture venues matched!")

Saved venue heat index!

Fixture venues not in heat table: 0
All fixture venues matched!


## Team Heat Tolerance

### How values are assigned (scale 1-10)

Heat tolerance is based on the **average summer temperature of each team's
home country**. The logic is simple — players who grew up and train in hot
climates are physiologically better adapted to performing in heat. Their
bodies are more efficient at cooling, they sweat more effectively, and they
recover faster in high temperatures.

This is supported by sports science research showing that:
- Heat acclimatisation takes 10-14 days of exposure to hot conditions
- Players from hot climates maintain higher work rates in heat vs cold-climate players
- Core body temperature rises faster in unacclimatised athletes

### Scoring guide

| Score | Climate type | Example countries |
|-------|-------------|-------------------|
| 9-10  | Hot/tropical — avg summer 35°C+ | Saudi Arabia, Iraq, Nigeria, Senegal |
| 7-8   | Warm — avg summer 28-35°C | Brazil, Mexico, Morocco, Iran |
| 5-6   | Moderate — avg summer 20-28°C | Spain, USA, Japan, Argentina |
| 3-4   | Cool — avg summer 15-20°C | France, Germany, Croatia |
| 1-2   | Cold — avg summer below 15°C | England, Poland, Denmark |

### Why this won't overfit
- It is one single integer feature per team — not complex
- It only interacts with venue heat index, not the full model
- The signal is directional and consistent — cold climate teams
  always struggle more in extreme heat, never less
- Historical evidence supports this: European teams have historically
  underperformed in hot World Cup venues (Brazil 2014, Qatar 2022)

### Limitation
Individual players may differ from their national average — for example
many African players play club football in cold European leagues and may
be less heat adapted than their national score suggests. We treat this
as a team-level approximation, not an individual measure.

In [21]:
# Team heat tolerance ( scale 1-10)
# 10 = very accustomed to heat , 1 = cold climate team

# Final tolerance table with exact 48 WC team names
team_tolerance = {
    # High tolerance — hot/tropical climates
    "Saudi Arabia":         9, "Iraq":                 9, "Panama":               9,
    "Haiti":                9, "Ivory Coast":          9, "DR Congo":             9,
    "Egypt":                9, "Senegal":              9, "Ghana":                9,
    "Qatar":                9, "Jordan":               8, "Iran":                 8,
    "Morocco":              8, "Algeria":              8, "Tunisia":              8,
    "Cape Verde":           8, "Curaçao":              8, "Colombia":             7,
    "Brazil":               8, "Paraguay":             7, "Mexico":               7,
    "South Africa":         7,

    # Medium tolerance — mixed/Mediterranean
    "Spain":                6, "Portugal":             6, "Turkey":               6,
    "Ecuador":              6, "USA":                  6, "Australia":            6,
    "Japan":                6, "Uzbekistan":           6, "Uruguay":              5,
    "Argentina":            5, "South Korea":          5,

    # Low tolerance — cool/cold climates
    "France":               4, "Croatia":              4, 
    "Germany":              3, "Netherlands":          3, "Belgium":              3,
    "Switzerland":          3,  "Canada":              3,
    "New Zealand":          3, "Austria":              3, "Norway":               3,
    "Sweden":               3, "Scotland":             2, "Bosnia & Herzegovina": 2,
    "Czech Republic":       2, "England":              2,
}

tolerance_df = pd.DataFrame([
    {"team": team, "heat_tolerance": score}
    for team, score in team_tolerance.items()
]).sort_values("heat_tolerance", ascending=False).reset_index(drop=True)

print(f"Teams in tolerance table: {len(tolerance_df)}")


Teams in tolerance table: 48


In [22]:
tolerance_df["team"].value_counts().sum()

np.int64(48)

In [23]:
print(tolerance_df[tolerance_df["team"].duplicated(keep=False)].to_string(index=False))

Empty DataFrame
Columns: [team, heat_tolerance]
Index: []


In [24]:
# Scale heat index to 1-10 to match team tolerance scale

hi_min = venue_df["heat_index"].min()
hi_max = venue_df["heat_index"].max()

venue_df["heat_index_scaled"] = round(
    1 + 9 * (venue_df["heat_index"] - hi_min) / (hi_max - hi_min), 1
)

print("Venue heat index scaled:")
print(venue_df[["venue","heat_index","heat_index_scaled"]].to_string(index=False))

Venue heat index scaled:
                                venue  heat_index  heat_index_scaled
                              Houston        45.7               10.0
                Miami (Miami Gardens)        40.2                7.8
                   Dallas (Arlington)        39.3                7.4
                Monterrey (Guadalupe)        37.8                6.8
                              Atlanta        37.0                6.5
                          Kansas City        35.9                6.0
                         Philadelphia        31.8                4.4
New York/New Jersey (East Rutherford)        29.7                3.5
                  Boston (Foxborough)        28.3                3.0
                Guadalajara (Zapopan)        27.4                2.6
              Los Angeles (Inglewood)        26.7                2.3
                              Toronto        26.0                2.0
                          Mexico City        25.0                1.6
         

In [25]:
# Check what happened
print("fixture columns:", fixture.columns.tolist())
print("\nFirst few rows of tolerance_df:")
print(tolerance_df.head())
print("\nSample home teams in fixture:")
print(fixture["home_team"].head(10).tolist())

fixture columns: ['match_num', 'date', 'round', 'group', 'home_team', 'away_team', 'venue', 'home_score', 'away_score', 'stage']

First few rows of tolerance_df:
           team  heat_tolerance
0  Saudi Arabia               9
1          Iraq               9
2        Panama               9
3         Haiti               9
4   Ivory Coast               9

Sample home teams in fixture:
['Mexico', 'South Korea', 'Czech Republic', 'Mexico', 'Czech Republic', 'South Africa', 'Canada', 'Qatar', 'Switzerland', 'Canada']


### Combine venue heat index and team tolerance into one feature — the heat disadvantage score

In [26]:
venue_df.columns.to_list()

['venue', 'temp_c', 'humidity', 'heat_index', 'heat_index_scaled']

In [27]:
print("venue in fixture not in venue_df")
print([v for v in fixture["venue"].unique() if v not in venue_df["venue"].values])

venue in fixture not in venue_df
[]


### Merge all DataFrame

In [28]:
# Merge venue heat index onto fixture
fixture = fixture.merge(
    venue_df[["venue","heat_index_scaled"]],
    on = "venue",
    how = "left",
)

print("Columns of fixture:", fixture.columns.to_list())
print("\nMissing heat_index_scaled:", fixture["heat_index_scaled"].isna().sum())
print("\nSamples:")
print(fixture[["home_team","venue","heat_index_scaled"]].head(5).to_string(index=False))

Columns of fixture: ['match_num', 'date', 'round', 'group', 'home_team', 'away_team', 'venue', 'home_score', 'away_score', 'stage', 'heat_index_scaled']

Missing heat_index_scaled: 0

Samples:
     home_team                 venue  heat_index_scaled
        Mexico           Mexico City                1.6
   South Korea Guadalajara (Zapopan)                2.6
Czech Republic               Atlanta                6.5
        Mexico Guadalajara (Zapopan)                2.6
Czech Republic           Mexico City                1.6


In [29]:
# Merge home team tolerance
fixture = fixture.merge(
    tolerance_df.rename(columns={"team" : "home_team", "heat_tolerance" : "home_heat_tolerance"}),
    on = "home_team",
    how = "left",
)
print("Missing home_heat_tolerance:", fixture["home_heat_tolerance"].isna().sum())
print("\nSample:")
print(fixture[["home_team","heat_index_scaled","home_heat_tolerance"]].head(5).to_string(index=False))

Missing home_heat_tolerance: 32

Sample:
     home_team  heat_index_scaled  home_heat_tolerance
        Mexico                1.6                  7.0
   South Korea                2.6                  5.0
Czech Republic                6.5                  2.0
        Mexico                2.6                  7.0
Czech Republic                1.6                  2.0


In [30]:
# Find all teams missing from tolerance_df
home_teams = fixture["home_team"].unique()
misiing = [t for t in home_teams if t not in tolerance_df["team"].values]
print("Team missing from tolerance table:")
for t in sorted(missing):
    print(" ", t)

Team missing from tolerance table:


In [31]:
# Check where NaN in tolerance is coming from
print(fixture[fixture["home_heat_tolerance"].isna()][["match_num","stage","home_team","away_team"]].head(10).to_string(index = False))

 match_num       stage home_team  away_team
        73 Round of 32        2A         2B
        74 Round of 32        1E 3A/B/C/D/F
        75 Round of 32        1F         2C
        76 Round of 32        1C         2F
        77 Round of 32        1I 3C/D/F/G/H
        78 Round of 32        2E         2I
        79 Round of 32        1A 3C/E/F/H/I
        80 Round of 32        1L 3E/H/I/J/K
        81 Round of 32        1D 3B/E/F/I/J
        82 Round of 32        1G 3A/E/H/I/J


In [32]:
# Count real teams in fixture (exclude placeholders)
all_fixture_teams = pd.concat([fixture["home_team"], fixture["away_team"]]).unique()
real_teams = sorted([t for t in all_fixture_teams if not t.startswith(("L","W","1","2","3"))])
print(f"Real teams in fixture: {len(real_teams)}")
for t in real_teams:
    print(f"  {t}")

Real teams in fixture: 48
  Algeria
  Argentina
  Australia
  Austria
  Belgium
  Bosnia & Herzegovina
  Brazil
  Canada
  Cape Verde
  Colombia
  Croatia
  Curaçao
  Czech Republic
  DR Congo
  Ecuador
  Egypt
  England
  France
  Germany
  Ghana
  Haiti
  Iran
  Iraq
  Ivory Coast
  Japan
  Jordan
  Mexico
  Morocco
  Netherlands
  New Zealand
  Norway
  Panama
  Paraguay
  Portugal
  Qatar
  Saudi Arabia
  Scotland
  Senegal
  South Africa
  South Korea
  Spain
  Sweden
  Switzerland
  Tunisia
  Turkey
  USA
  Uruguay
  Uzbekistan


In [33]:
fixture.head()

,match_num,date,round,group,home_team,away_team,venue,home_score,away_score,stage,heat_index_scaled,home_heat_tolerance
0,1,2026-06-11,Matchday 1,Group A,Mexico,South Africa,Mexico City,None,None,Group,1.6,7.0
1,2,2026-06-11,Matchday 1,Group A,South Korea,Czech Republic,Guadalajara (Zapopan),None,None,Group,2.6,5.0
2,3,2026-06-18,Matchday 8,Group A,Czech Republic,South Africa,Atlanta,None,None,Group,6.5,2.0
3,4,2026-06-18,Matchday 8,Group A,Mexico,South Korea,Guadalajara (Zapopan),None,None,Group,2.6,7.0
4,5,2026-06-24,Matchday 14,Group A,Czech Republic,Mexico,Mexico City,None,None,Group,1.6,2.0


In [34]:
print(f"Teams in tolerance table: {len(tolerance_df)}")
extras  = [t for t in tolerance_df["team"].values if t not in real_teams]
missing = [t for t in real_teams if t not in tolerance_df["team"].values]
print(f"Extras:  {extras  if extras  else 'None'}")
print(f"Missing: {missing if missing else 'All 48 covered!'}")

Teams in tolerance table: 48
Extras:  None
Missing: All 48 covered!


In [35]:
print(fixture.shape)
print(fixture.columns.to_list())

(104, 12)
['match_num', 'date', 'round', 'group', 'home_team', 'away_team', 'venue', 'home_score', 'away_score', 'stage', 'heat_index_scaled', 'home_heat_tolerance']


In [36]:
# Merge away team tolerance
fixture = fixture.merge(
    tolerance_df.rename(columns={"team":"away_team", "heat_tolerance":"away_heat_tolerance"}),
    on = "away_team",
    how = "left",
)
print("Missing away_heat_tolerance:", fixture["away_heat_tolerance"].isna().sum())
print("\nSample:")
print(fixture[["home_team", "away_team", "heat_index_scaled", 
               "home_heat_tolerance", "away_heat_tolerance"]].head(5).to_string(index=False))
 

Missing away_heat_tolerance: 32

Sample:
     home_team      away_team  heat_index_scaled  home_heat_tolerance  away_heat_tolerance
        Mexico   South Africa                1.6                  7.0                  7.0
   South Korea Czech Republic                2.6                  5.0                  2.0
Czech Republic   South Africa                6.5                  2.0                  7.0
        Mexico    South Korea                2.6                  7.0                  5.0
Czech Republic         Mexico                1.6                  2.0                  7.0


In [37]:
fixture.head()

,match_num,date,round,group,home_team,away_team,venue,home_score,away_score,stage,heat_index_scaled,home_heat_tolerance,away_heat_tolerance
0,1,2026-06-11,Matchday 1,Group A,Mexico,South Africa,Mexico City,None,None,Group,1.6,7.0,7.0
1,2,2026-06-11,Matchday 1,Group A,South Korea,Czech Republic,Guadalajara (Zapopan),None,None,Group,2.6,5.0,2.0
2,3,2026-06-18,Matchday 8,Group A,Czech Republic,South Africa,Atlanta,None,None,Group,6.5,2.0,7.0
3,4,2026-06-18,Matchday 8,Group A,Mexico,South Korea,Guadalajara (Zapopan),None,None,Group,2.6,7.0,5.0
4,5,2026-06-24,Matchday 14,Group A,Czech Republic,Mexico,Mexico City,None,None,Group,1.6,2.0,7.0


In [38]:
# Save final fixture table
processed_dir = Path.home() / "wc2026" / "data" / "processed"

fixture.to_csv(processed_dir / "fixtures_2026.csv", index=False)

print("Saved!")
print("Shape:", fixture.shape)
print("Columns:", fixture.columns.tolist())

Saved!
Shape: (104, 13)
Columns: ['match_num', 'date', 'round', 'group', 'home_team', 'away_team', 'venue', 'home_score', 'away_score', 'stage', 'heat_index_scaled', 'home_heat_tolerance', 'away_heat_tolerance']


### FIFA ranking points

In [39]:
import os

# List files in processed dir first
print("Files in processed dir:")
for f in processed_dir.iterdir():
    print(f" {f.name}")

Files in processed dir:
 fifa_rankings_2026.csv
 fixtures_2026.csv
 fixture_2026.csv
 venue_heat_index.csv


In [40]:
# Delete old/duplicate files
files_to_delete = [
    processed_dir / "fifa_rankings_2026.csv",
    processed_dir / "fixture_2026.csv",
]
for f in files_to_delete:
    if f.exists():
        os.remove(f)
        print(f"Deleted: {f.name}")

# Confirm
print("\nFiles remaining:")
for f in processed_dir.iterdir():
    print(f" {f.name}")

Deleted: fifa_rankings_2026.csv
Deleted: fixture_2026.csv

Files remaining:
 fixtures_2026.csv
 venue_heat_index.csv


In [41]:
# Build FIFA 2026 rankings DataFrame with all columns
fifa_2026_data = [
    (1,  "Argentina",            1874.81, "CONMEBOL"),
    (2,  "Spain",                1876.40, "UEFA"),
    (3,  "France",               1877.32, "UEFA"),
    (4,  "England",              1825.97, "UEFA"),
    (5,  "Portugal",             1763.83, "UEFA"),
    (6,  "Brazil",               1761.16, "CONMEBOL"),
    (7,  "Morocco",              1755.87, "CAF"),
    (8,  "Netherlands",          1757.87, "UEFA"),
    (9,  "Belgium",              1734.71, "UEFA"),
    (10, "Germany",              1730.37, "UEFA"),
    (11, "Croatia",              1721.40, "UEFA"),
    (13, "Colombia",             1664.20, "CONMEBOL"),
    (14, "Mexico",               1661.11, "CONCACAF"),
    (15, "Senegal",              1650.50, "CAF"),
    (16, "Uruguay",              1648.30, "CONMEBOL"),
    (17, "USA",                  1642.59, "CONCACAF"),
    (18, "Japan",                1621.88, "AFC"),
    (19, "Switzerland",          1617.24, "UEFA"),
    (21, "Iran",                 1613.96, "AFC"),
    (22, "Turkey",               1600.03, "UEFA"),
    (23, "Ecuador",              1597.40, "CONMEBOL"),
    (24, "Austria",              1570.20, "UEFA"),
    (25, "South Korea",          1564.35, "AFC"),
    (27, "Australia",            1557.30, "AFC"),
    (28, "Algeria",              1546.75, "CAF"),
    (29, "Egypt",                1540.20, "CAF"),
    (30, "Canada",               1522.44, "CONCACAF"),
    (31, "Norway",               1519.80, "UEFA"),
    (33, "Ivory Coast",          1500.22, "CAF"),
    (34, "Panama",               1497.50, "CONCACAF"),
    (38, "Sweden",               1478.40, "UEFA"),
    (39, "Czech Republic",       1475.10, "UEFA"),
    (40, "Paraguay",             1461.30, "CONMEBOL"),
    (42, "Scotland",             1454.80, "UEFA"),
    (45, "DR Congo",             1444.20, "CAF"),
    (46, "Tunisia",              1441.90, "CAF"),
    (50, "Uzbekistan",           1425.26, "AFC"),
    (56, "Iraq",                 1409.60, "AFC"),
    (57, "Qatar",                1407.30, "AFC"),
    (60, "South Africa",         1395.15, "CAF"),
    (61, "Saudi Arabia",         1392.40, "AFC"),
    (63, "Jordan",               1380.75, "AFC"),
    (64, "Bosnia & Herzegovina", 1375.40, "UEFA"),
    (67, "Cape Verde",           1364.50, "CAF"),
    (73, "Ghana",                1342.10, "CAF"),
    (82, "Curaçao",              1311.50, "CONCACAF"),
    (83, "Haiti",                1309.20, "CONCACAF"),
    (85, "New Zealand",          1301.10, "OFC"),
]

fifa_2026_df = pd.DataFrame(fifa_2026_data, 
                             columns=["fifa_rank", "team", "fifa_points", "confederation"])

print("Shape:", fifa_2026_df.shape)
print("\nMissing WC teams:")
missing = [t for t in real_teams if t not in fifa_2026_df["team"].values]
print(missing if missing else "All 48 covered!")
print("\nTop 10:")
print(fifa_2026_df.head(10).to_string(index=False))

Shape: (48, 4)

Missing WC teams:
All 48 covered!

Top 10:
 fifa_rank        team  fifa_points confederation
         1   Argentina      1874.81      CONMEBOL
         2       Spain      1876.40          UEFA
         3      France      1877.32          UEFA
         4     England      1825.97          UEFA
         5    Portugal      1763.83          UEFA
         6      Brazil      1761.16      CONMEBOL
         7     Morocco      1755.87           CAF
         8 Netherlands      1757.87          UEFA
         9     Belgium      1734.71          UEFA
        10     Germany      1730.37          UEFA


In [42]:
# Save FIFA 2026 rankings
fifa_2026_df.to_csv(processed_dir / "fifa_rankings_2026.csv", index=False)

print("Saved!")
print("\nFiles in processed dir:")
for f in processed_dir.iterdir():
    print(f"  {f.name}")

Saved!

Files in processed dir:
  fifa_rankings_2026.csv
  fixtures_2026.csv
  venue_heat_index.csv


## Compute Elo ratings from historical results

## Elo Rating System

### What is Elo?
Elo is a method for calculating the relative skill of players or teams.
Originally developed for chess by Arpad Elo, it is now widely used in
football, basketball, and other sports. The World Football Elo Ratings
have been tracking national teams since 1872.

### How it works
Every team starts with a rating of 1500. After each match, points are
transferred from the loser to the winner. The amount transferred depends
on three things:

1. **The result** — win, draw or loss
2. **The expected result** — beating a much stronger team earns more points
   than beating a weaker one
3. **The match importance** — World Cup matches transfer more points than friendlies

### The formula

- **K factor** — controls how much each match moves the rating
  - World Cup: 60 (highest importance)
  - Competitive: 40
  - Friendly: 20 (down-weighted — lower effort, noisier results)
- **Actual Result** — 1 for win, 0.5 for draw, 0 for loss
- **Expected Result** — probability of winning based on rating difference

### Why Elo is better than FIFA rankings for our model
| | FIFA Rankings | Elo |
|---|---|---|
| Update frequency | ~6 times per year | After every match |
| Historical coverage | From 1992 | From 1872 |
| Gaps | Yes (Sep 2024 - Jun 2026) | No gaps ever |
| Accounts for match importance | Yes | Yes |
| Self-contained | No (external source) | Yes (we compute it) |

### Home advantage
In a normal international match, the home team gets a +100 Elo point
adjustment before calculating expected result. In the 2026 World Cup,
only USA, Mexico and Canada receive this bonus — all other matches
are played at neutral venues.

### Why this matters for prediction
The Elo difference between two teams is one of the strongest predictors
of expected goals. A team rated 200 points higher wins roughly 75% of
matches historically. This becomes the core strength feature in our
Poisson goals model.

In [43]:
# Load the results spine 
results = pd.read_csv(raw_dir / "results.csv", parse_dates=["date"])

# Only played matches
played = results[results["home_score"].notna()].sort_values("date").copy()

print("Total match played:", len(played))
print("Date range", played.date.min().date(), "->", played.date.max().date())


Total match played: 49405
Date range 1872-11-30 -> 2026-06-10


In [44]:
import os
print("Files in raw dir:")
for f in raw_dir.iterdir():
    print(f"  {f.name}")

Files in raw dir:
  results.csv
  wc2026_fixture.json
  fifa_rankings.csv


In [54]:
# Compute Elo ratings

# Parameters
ELO_DEFAULT = 1500  # starting rating for every team
HOME_ADV    = 100   # home team gets +100 points advantage
K_WC        = 60    # World Cup matches weighted highest
K_COMP      = 40    # all other competitive matches
K_FRIENDLY  = 20    # friendlies weighted lowest (noisier games)

def get_k(tournament):
    if tournament == "FIFA World Cup":
        return K_WC
    if "Friendly" in tournament:
        return K_FRIENDLY
    return K_COMP

def expected_score(ra,rb):
    # Probability team A beats team B
    return 1 / (1+10 ** ((rb-ra)/400))

def match_outcome(home_goals, away_goals):
    if home_goals > away_goals:
        return 1.0    # Home win
    if home_goals == away_goals:
        return 0.5    # Draw
    return 0.0        # away win

# Compute elo
# store current ratings for every team
ratings = {}

for row in played.itertuples(index=False):
    ht , at = row.home_team, row.away_team

    # Get current ratings for every team
    rh = ratings.get(ht, ELO_DEFAULT)
    ra = ratings.get(at, ELO_DEFAULT)

    # Aplly home advantage
    adj_rh = rh + (0 if row.neutral else HOME_ADV)

    # Expected and actual outcome
    E_h = expected_score(adj_rh, ra)
    S_h = match_outcome(int(row.home_score), int(row.away_score))
    K = get_k(row.tournament)

    # Update ratings
    ratings[ht] = rh + K * (S_h - E_h)
    ratings[at] = ra + K * ((1 - S_h) - (1 - E_h))

print(f"Ratings computed for {len(ratings)} teams")

Ratings computed for 336 teams


In [55]:
# Convert Elo ratings to DataFrame

elo_df = pd.DataFrame([
    {"team":team, "elo_rating": round(rating, 1)}
    for team, rating in ratings.items()
]).sort_values("elo_rating", ascending=False).reset_index(drop=True)

elo_df.index += 1
elo_df.index.name = "elo_rank"

print("Top 20 teams by Elo:")
print(elo_df.head(20).to_string())

print("\nWC48 teams:")
missing = [t for t in real_teams if t not in elo_df["team"].values]
print(f"MIssing: {missing if missing else 'All 48 covered'}")

Top 20 teams by Elo:
                 team  elo_rating
elo_rank                         
1               Spain      2080.2
2           Argentina      2073.6
3              France      2017.1
4            Portugal      1962.7
5             England      1960.9
6              Brazil      1956.1
7            Colombia      1946.1
8             Ecuador      1937.8
9             Germany      1921.4
10        Netherlands      1915.8
11              Japan      1911.3
12            Morocco      1907.3
13            Croatia      1899.7
14             Mexico      1898.6
15            Uruguay      1891.4
16             Turkey      1891.0
17              Italy      1883.5
18            Belgium      1866.4
19        Switzerland      1852.5
20           Paraguay      1851.5

WC48 teams:
MIssing: ['Bosnia & Herzegovina', 'USA']


In [56]:
# Find the actual name used in result spine
for name in ["Bosnia", "United States", "USA"]:
    matches = results[results["home_team"].str.contains(name, case=False)]["home_team"].unique()
    print(f"{name}: {matches}")

Bosnia: <ArrowStringArray>
['Bosnia and Herzegovina']
Length: 1, dtype: str
United States: <ArrowStringArray>
['United States', 'United States Virgin Islands']
Length: 2, dtype: str
USA: <ArrowStringArray>
[]
Length: 0, dtype: str


In [59]:
# Fix names in ratings dictionary
ratings["Bosnia & Herzegovina"] = ratings.pop("Bosnia and Herzegovina")
ratings["USA"] = ratings.pop("United States")

# Rebuild elo_df
elo_df = pd.DataFrame([
    {"team": team, "elo_rating": round(rating, 1)}
    for team, rating in ratings.items()
]).sort_values("elo_rating", ascending=False).reset_index(drop=True)

elo_df.index += 1
elo_df.index.name = "elo_rank"

# Check again
missing = [t for t in real_teams if t not in elo_df["team"].values]
print(f"Missing: {missing if missing else 'All 48 covered!'}")

print("\nUSA and Bosnia ratings:")
print(elo_df[elo_df["team"].isin(["USA", "Bosnia & Herzegovina"])].to_string())

Missing: All 48 covered!

USA and Bosnia ratings:
                          team  elo_rating
elo_rank                                  
39                         USA      1756.1
83        Bosnia & Herzegovina      1624.3


In [60]:
# Save Elo ratings
elo_df.reset_index().to_csv(processed_dir / "elo_ratings_2026.csv", index=False)

print("Saved!")
print("\nFiles in processed dir:")
for f in processed_dir.iterdir():
    print(f"  {f.name}")

Saved!

Files in processed dir:
  fifa_rankings_2026.csv
  fixtures_2026.csv
  venue_heat_index.csv
  elo_ratings_2026.csv


## Real Transfer market data - Squad value

In [61]:
# Squad market values (Transfermarkt, June 2026)

squad_values_raw = {
    "France":                   1520,
    "England":                  1360,
    "Spain":                    1220,
    "Portugal":                 1010,
    "Germany":                   947,
    "Brazil":                    928,
    "Argentina":                 782,
    "Netherlands":               754,
    "Norway":                    590,
    "Belgium":                   548,
    "Ivory Coast":               522,
    "Senegal":                   478,
    "Turkey":                    474,
    "Morocco":                   448,
    "Sweden":                    406,
    "Croatia":                   387,
    "USA":                       386,
    "Ecuador":                   369,
    "Uruguay":                   359,
    "Switzerland":               333,
    "Colombia":                  302,
    "Japan":                     271,
    "Algeria":                   257,
    "Austria":                   245,
    "Ghana":                     235,
    "Canada":                    199,
    "Mexico":                    192,
    "Czech Republic":            188,
    "Scotland":                  170,
    "Paraguay":                  154,
    "Bosnia & Herzegovina":      152,
    "DR Congo":                  144,
    "South Korea":               139,
    "Egypt":                     116,
    "Uzbekistan":                 85,
    "Australia":                  77,
    "Tunisia":                    70,
    "Haiti":                      56,
    "Cape Verde":                 55,
    "South Africa":               49,
    "Saudi Arabia":               41,
    "Panama":                     35,
    "New Zealand":                34,
    "Iran":                       32,
    "Curaçao":                    26,
    "Iraq":                       21,
    "Jordan":                     20,
    "Qatar":                      20,
}

squad_df = pd.DataFrame([
    {"team": team, "squad_value_eur_m": value}
    for team, value in squad_values_raw.items()
]).sort_values("squad_value_eur_m", ascending=False).reset_index(drop=True)

squad_df.index += 1
squad_df.index.name = "value_rank"

print(f"Total teams: {len(squad_df)}")
missing = [t for t in real_teams if t not in squad_df["team"].values]
print(f"Missing: {missing if missing else 'All 48 covered!'}")

print("\nTop 10 squads by value:")
print(squad_df.head(10).to_string())

Total teams: 48
Missing: All 48 covered!

Top 10 squads by value:
                   team  squad_value_eur_m
value_rank                                
1                France               1520
2               England               1360
3                 Spain               1220
4              Portugal               1010
5               Germany                947
6                Brazil                928
7             Argentina                782
8           Netherlands                754
9                Norway                590
10              Belgium                548


In [62]:
# Save squad values
squad_df.reset_index().to_csv(processed_dir / "squad_values_2026.csv", index=False)

print("Saved!")
print("\nAll files in processed dir:")
for f in sorted(processed_dir.iterdir()):
    print(f"  {f.name}")

Saved!

All files in processed dir:
  elo_ratings_2026.csv
  fifa_rankings_2026.csv
  fixtures_2026.csv
  squad_values_2026.csv
  venue_heat_index.csv


In [63]:
# Data collection summary
print("=" * 50)
print("DATA COLLECTION COMPLETE")
print("=" * 50)

print("\n RAW DATA:")
for f in sorted(raw_dir.iterdir()):
    size = f.stat().st_size / 1024
    print(f"  {f.name:<30} {size:.1f} KB")

print("\n PROCESSED DATA:")
for f in sorted(processed_dir.iterdir()):
    df = pd.read_csv(f)
    print(f"  {f.name:<35} {df.shape[0]} rows x {df.shape[1]} cols")

print("\n DATASETS READY:")
print("  ✓ Historical results     — 49,400 matches since 1872")
print("  ✓ WC26 fixtures          — 104 matches with weather features")
print("  ✓ FIFA rankings 2026     — 48 teams, June 2026")
print("  ✓ Elo ratings            — 48 teams, self computed")
print("  ✓ Squad values           — 48 teams, Transfermarkt June 2026")
print("  ✓ Venue heat index       — 16 venues")
print("\n NEXT: Feature Engineering")

DATA COLLECTION COMPLETE

 RAW DATA:
  fifa_rankings.csv              2654.9 KB
  results.csv                    3637.1 KB
  wc2026_fixture.json            19.5 KB

 PROCESSED DATA:
  elo_ratings_2026.csv                336 rows x 3 cols
  fifa_rankings_2026.csv              48 rows x 4 cols
  fixtures_2026.csv                   104 rows x 13 cols
  squad_values_2026.csv               48 rows x 3 cols
  venue_heat_index.csv                16 rows x 4 cols

 DATASETS READY:
  ✓ Historical results     — 49,400 matches since 1872
  ✓ WC26 fixtures          — 104 matches with weather features
  ✓ FIFA rankings 2026     — 48 teams, June 2026
  ✓ Elo ratings            — 48 teams, self computed
  ✓ Squad values           — 48 teams, Transfermarkt June 2026
  ✓ Venue heat index       — 16 venues

 NEXT: Feature Engineering
